# Acoustic Receptions

In [ ]:
from n23_fluctuations_ragland import tools
from matplotlib import pyplot as plt
import numpy as np
import xarray as xr
from dotenv import load_dotenv, find_dotenv
import os
import cmps
import colorsys
import pickle

_ = load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'

## Plot Settings

In [ ]:
# Define your colors
whoi_colors = ['#00B7BD', '#EE5340']

plt.rcParams.update({"font.size": 8})
plt.rcParams.update({"text.usetex": True})

## Open Data

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/mean_pulse_lower.nc"
mean_pulse_lower = xr.open_dataarray(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/mean_pulse_upper.nc"
mean_pulse_upper = xr.open_dataarray(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/mean_pulse_lower_noise.nc"
mean_pulse_lower_noise = xr.open_dataarray(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/mean_pulse_upper_noise.nc"
mean_pulse_upper_noise = xr.open_dataarray(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"
with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

In [ ]:
rays = ray_run['rays']
eigs = ray_run['eigs']

In [ ]:
dss = tools.open_receptions()

In [ ]:
Tx_idx = 12

single_lower = dss[Tx_idx][32:].swap_dims({"hydrophone": "nominal_depth"}).rename({"nominal_depth": "depth"})
single_upper = dss[Tx_idx][:32].swap_dims({"hydrophone": "nominal_depth"}).rename({"nominal_depth": "depth"})

single_lower_noise = 20 * np.log10(
    np.sqrt(np.abs(single_lower.sel({"time": slice(90, 95)}) ** 2).mean(dim="time"))
)
single_upper_noise = 20 * np.log10(
    np.sqrt(np.abs(single_upper.sel({"time": slice(90, 95)}) ** 2).mean(dim="time"))
)

In [ ]:
thetas = np.linspace(-40, 40, 200)

bf_lower = tools.ULA_conventional_beamform(
    dss[Tx_idx][32:]
    .swap_dims({"hydrophone": "nominal_depth"})
    .rename({"nominal_depth": "depth"}),
    thetas,
)

bf_lower_noise = 20 * np.log10(
    np.sqrt(np.abs(bf_lower.sel({"time": slice(90, 95)}) ** 2).mean(dim="time"))
)

bf_upper = tools.ULA_conventional_beamform(
    dss[Tx_idx][:32]
    .swap_dims({"hydrophone": "nominal_depth"})
    .rename({"nominal_depth": "depth"}),
    thetas,
)

bf_upper_noise = 20 * np.log10(
    np.sqrt(np.abs(bf_upper.sel({"time": slice(90, 95)}) ** 2).mean(dim="time"))
)

## Create Plot *new*

In [ ]:
fig = plt.figure(figsize=(6.67,4))
snr_bounds = [4,35]


plt.subplot(2,1,1)
(20*np.log10(mean_pulse_lower) - mean_pulse_lower_noise).sel({'depth':slice(950, 1250)}).plot(cmap=cmps.mako_r_white(), vmin=5, vmax=30, cbar_kwargs={'label':'SNR [dB]'}, rasterized=True)

plt.xlim([100.25,101.75])
plt.ylim([1250,950])
plt.ylabel('depth [m]')
plt.xlabel('')
plt.gca().set_xticklabels('')
plt.title('Acoustic Reception: Mean Pulse', loc='left', pad=-2)

plt.subplot(2,1,2)

im2 = (20*np.log10(np.abs(bf_lower)) - bf_lower_noise).plot(x='time', cmap=cmps.mako_r_white(), vmin=7, vmax=40, cbar_kwargs={'label':'SNR [dB]'}, rasterized=True)
plt.xlim([100.25,101.75])
plt.ylim([-25,25])
plt.xlabel('time [s]')
plt.ylabel('angle [deg]')
plt.title('Single Beamformed Reception', loc='left', pad=-2)

plt.tight_layout(pad=0.25)

fig.savefig('../../figures/Acoustic_Receptions.pdf', dpi=500)

## Create Plot

In [ ]:
fig = plt.figure(figsize=(6.67,4))

plt.subplot(2,1,1)

(20*np.log10(np.abs(mean_pulse_upper)) - mean_pulse_upper_noise).sel({'depth':slice(200, 475)}).plot(add_colorbar=False, vmax=35, vmin=3, cmap=cmps.mako_r_white(), rasterized=True)


plt.xlim([100,115])
plt.ylim([475,200])
plt.ylabel('depth [m]')
plt.xlabel('')
plt.gca().set_xticklabels('')
plt.yticks(np.arange(250, 499, 100))

plt.subplot(2,1,2)
im2 = (20*np.log10(np.abs(mean_pulse_lower)) - mean_pulse_lower_noise).sel({'depth':slice(950, 1250)}).plot(vmax=35, vmin=3, cmap=cmps.mako_r_white(), add_colorbar=False, rasterized=True)


plt.yticks(np.arange(1201,1000,-100))
plt.xlim([100,115])
plt.ylim([1250,950])

plt.ylabel('depth [m]')
plt.xlabel('time [s]')

plt.tight_layout(pad=1)

# Shift subplots left and add shared colorbar on the right
plt.subplots_adjust(right=0.87)
cbar_ax = fig.add_axes([0.88, 0.2, 0.02, 0.7])
fig.colorbar(im2, cax=cbar_ax, label='SNR [dB]')

#fig.savefig('../../figures/Acoustic_Receptions.pdf', bbox_inches='tight', dpi=500)

In [ ]:
fig = plt.figure(figsize=(6.67,4))

snr_bounds = (5,35)
snr_bounds_bf = (5,35)

plt.subplot(2,1,1)

(20*np.log10(np.abs(single_upper)) - single_upper_noise).plot(add_colorbar=False, vmax=snr_bounds[1], vmin=snr_bounds[0], cmap=cmps.mako_r_white(), rasterized=True)
plt.title('Upper Array', loc='left')

plt.xlim([100,115])
plt.ylim([single_upper.depth[-1], single_upper.depth[0]])

plt.ylabel('depth [m]')
plt.xlabel('')
plt.gca().set_xticklabels('')
#plt.yticks(np.arange(250, 499, 100))

plt.subplot(2,1,2)
im2 = (20*np.log10(np.abs(single_lower)) - single_lower_noise).plot(vmax=snr_bounds[1], vmin=snr_bounds[0], cmap=cmps.mako_r_white(), add_colorbar=False, rasterized=True)
plt.title('Lower Array', loc='left')

#plt.yticks(np.arange(1201,1000,-100))
plt.xlim([100,115])
plt.ylim([single_lower.depth[-1], single_lower.depth[0]])

plt.ylabel('depth [m]')
plt.xlabel('time [s]')

plt.tight_layout(pad=1)

# Shift subplots left and add shared colorbar on the right
plt.subplots_adjust(right=0.87)
cbar_ax = fig.add_axes([0.88, 0.2, 0.02, 0.7])
fig.colorbar(im2, cax=cbar_ax, label='SNR [dB]')

fig.savefig('../../figures/TimeFront_Reception.pdf', bbox_inches='tight', dpi=500)

In [ ]:
fig = plt.figure(figsize=(6.67,4))

snr_bounds = (5,35)

plt.subplot(2,1,1)

(20*np.log10(np.abs(bf_upper)) - bf_upper_noise).plot(x='time', add_colorbar=False, vmax=snr_bounds[1], vmin=snr_bounds[0], cmap=cmps.mako_r_white(), rasterized=True)
plt.title('Upper Array',loc='left')

for ray_id in ray_ids:
    ray_id_pr,_ = tools.parse_ray_id(ray_id)
    eig_idx = np.where(eigs.ray_id[1] == ray_id_pr)[0][0]

    plt.scatter(x=eigs.ts[1][eig_idx,-1]-1.8, y=eigs.received_angles[1][eig_idx], s=85, facecolors='none', edgecolors=whoi_colors[1], alpha=0.8, linewidths=1)

    
plt.xlim([100,112])
plt.ylim([-40,40])

plt.ylabel('received angle [deg]')
plt.xlabel('')
plt.gca().set_xticklabels('')
#plt.yticks(np.arange(250, 499, 100))

plt.subplot(2,1,2)
im2 = (20*np.log10(np.abs(bf_lower)) - bf_lower_noise).plot(x='time', vmax=snr_bounds[1], vmin=snr_bounds[0], cmap=cmps.mako_r_white(), add_colorbar=False, rasterized=True)

for ray_id in ray_ids:
    ray_id_pr,_ = tools.parse_ray_id(ray_id)
    eig_idx = np.where(eigs.ray_id[0] == ray_id_pr)[0][tools.EIG_INDEXS[ray_id][0][0]]

    plt.scatter(x=eigs.ts[0][eig_idx,-1]-1.8, y=eigs.received_angles[0][eig_idx], s=85, facecolors='none', edgecolors=whoi_colors[1], alpha=0.8, linewidths=1)


plt.title('')
plt.title('Lower Array',loc='left')

#plt.yticks(np.arange(1201,1000,-100))
plt.xlim([100,112])
#plt.ylim([single_lower.depth[-1], single_lower.depth[0]])

plt.ylabel('received angle [deg]')
plt.xlabel('time [s]')

plt.tight_layout(pad=1)

# Shift subplots left and add shared colorbar on the right
plt.subplots_adjust(right=0.87)
cbar_ax = fig.add_axes([0.88, 0.2, 0.02, 0.7])
fig.colorbar(im2, cax=cbar_ax, label='SNR [dB]')

fig.savefig('../../figures/Beamformed_Reception.pdf', bbox_inches='tight', dpi=500)

In [ ]:
fig = plt.figure(figsize=(3.37,2))

(20*np.log10(np.abs(bf_lower)) - bf_lower_noise).plot(x='time', cmap=cmps.mako_r_white(), vmin=8, vmax=35, cbar_kwargs={'label':'SNR [dB]'}, rasterized=True)


for ray_id in ray_ids:
    ray_id_pr,_ = tools.parse_ray_id(ray_id)
    eig_idx = np.where(eigs.ray_id[0] == ray_id_pr)[0][tools.EIG_INDEXS[ray_id][0][0]]

    plt.scatter(x=eigs.ts[0][eig_idx,-1]-1.9, y=eigs.received_angles[0][eig_idx], s=65, facecolors='none', edgecolors=whoi_colors[1], alpha=0.8, linewidths=1.5)

    #if eigs.ts[0][eig_idx,-1]-1.9 < 101.5:
    #    plt.text(x=eigs.ts[0][eig_idx,-1]-1.9 + 0.05, y=eigs.received_angles[0][eig_idx] + 1, s=ray_id)
    
plt.ylabel('angle [ degrees ]')
plt.xlabel('time [s]')


plt.xlim([100.25,101.5])
plt.ylim([-25,25])

plt.tight_layout(pad=0)
fig.savefig('../../figures/Beamformed_Lower_Ducted.pdf', bbox_inches='tight', dpi=500)

In [ ]:
fig = plt.figure(figsize=(3.37,2))

(20*np.log10(np.abs(single_lower)) - single_lower_noise).plot(x='time', cmap=cmps.mako_r_white(), vmin=8, vmax=35, rasterized=True, cbar_kwargs={'label':'SNR [dB]'})

plt.scatter(x=rays.ts[:,-1]-1.9, y=-rays.zs[:,-1], s=7, lw=0, c=whoi_colors[1], alpha=0.7)
#plt.plot(rays.ts[:,-1]-1.9, -rays.zs[:,-1], c=whoi_colors[1], alpha=0.7)

plt.ylabel('depth [ m ]')
plt.xlabel('time [s]')


plt.xlim([100.25,101.5])
plt.ylim([single_lower.depth[-1], single_lower.depth[0]])

plt.tight_layout(pad=0)
fig.savefig('../../figures/Timefront_Lower_Ducted.pdf', bbox_inches='tight', dpi=500)